# 📐 03. Rule-Based Baseline Detector
Thử nghiệm và tinh chỉnh các ngưỡng động học (deceleration, angle change, IoU) cho Rule-Based MVP.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from src.features.motion_features import MotionFeatureExtractor
from src.classifiers.rule_based import RuleBasedAccidentDetector

extractor = MotionFeatureExtractor(smoothing_window=5)
rule_detector = RuleBasedAccidentDetector(
    sudden_decel_thresh=15.0,
    direction_change_thresh=45.0,
    iou_overlap_thresh=0.15,
    alert_threshold=0.50
)
print("Rule-based baseline initialized!")

## 1. Mô phỏng va chạm giữa 2 xe máy

In [ ]:
# Tạo chuyển động xe 1 (chạy ngang sang phải rồi dừng đột ngột)
centers_a = np.array([[100 + i * 15, 200] for i in range(8)] + [[220, 200], [220, 200]], dtype=np.float32)
bboxes_a = np.array([[c[0]-20, c[1]-20, c[0]+20, c[1]+20] for c in centers_a], dtype=np.float32)

# Tạo chuyển động xe 2 (chạy ngược chiều sang trái rồi đâm vào xe 1)
centers_b = np.array([[340 - i * 15, 200] for i in range(8)] + [[225, 200], [225, 200]], dtype=np.float32)
bboxes_b = np.array([[c[0]-20, c[1]-20, c[0]+20, c[1]+20] for c in centers_b], dtype=np.float32)

feat_a = extractor.compute_single_track_features(centers_a, bboxes_a)
feat_b = extractor.compute_single_track_features(centers_b, bboxes_b)

print("Max Deceleration A:", np.max(feat_a["accel_mag"]))
print("Max Deceleration B:", np.max(feat_b["accel_mag"]))

## 2. Đánh giá quy luật va chạm

In [ ]:
result = rule_detector.evaluate_pairwise(
    tid_a=1, tid_b=2,
    feat_a=feat_a, feat_b=feat_b,
    bboxes_a=bboxes_a, bboxes_b=bboxes_b,
    current_frame=9
)

print("Rule Evaluation Result:", result)
if result:
    print(f"🚨 Alert Triggered! Score: {result['score']:.2f} | Reason: {result['reasons']}")